# 03 - Huấn luyện mô hình (BỔ SUNG Cross-Validation + Dummy Baseline)
BTL môn Trí tuệ nhân tạo — Dữ liệu mô phỏng

**Bổ sung so với bản cũ:**
1. Thêm mô hình **Dummy Baseline** (luôn đoán lớp đa số) làm mốc tham chiếu
2. Thêm **K-Fold Cross-Validation (5-fold)** — báo cáo trung bình ± độ lệch chuẩn, thay vì chỉ đánh giá qua 1 lần chia Train/Test duy nhất

## Bước 1: Load lại dữ liệu (đã xử lý đúng, không rò rỉ)

In [1]:
import joblib

X_train, X_test, y_train, y_test = joblib.load("../data/processed/train_test_data.pkl")
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (1200, 10) Test: (300, 10)


## Bước 2: Khởi tạo mô hình — thêm Dummy Baseline

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier

models = {
    "Dummy_Baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic_Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Decision_Tree": DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced'),
    "Random_Forest": RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": SVC(kernel='linear', probability=True, random_state=42, class_weight='balanced'),
}

## Bước 3: Cross-Validation 5-fold trên tập Train
Đánh giá độ ổn định của từng mô hình qua nhiều lần chia dữ liệu khác nhau, thay vì chỉ tin vào 1 lần chia duy nhất — đúng nguyên tắc "báo cáo cả trung bình và độ phân tán qua các lần gấp của kiểm chứng chéo".

In [3]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
import pandas as pd

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for name, model in models.items():
    scores_f1 = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1')
    scores_acc = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    cv_results.append({
        "Model": name,
        "F1_mean": round(scores_f1.mean(), 3),
        "F1_std": round(scores_f1.std(), 3),
        "Accuracy_mean": round(scores_acc.mean(), 3),
        "Accuracy_std": round(scores_acc.std(), 3),
    })
    print(f"{name}: F1 = {scores_f1.mean():.3f} ± {scores_f1.std():.3f} | "
          f"Accuracy = {scores_acc.mean():.3f} ± {scores_acc.std():.3f}")

cv_results_df = pd.DataFrame(cv_results)
cv_results_df.to_csv("../results/cv_results.csv", index=False)

Dummy_Baseline: F1 = 0.788 ± 0.000 | Accuracy = 0.650 ± 0.000
Logistic_Regression: F1 = 0.867 ± 0.013 | Accuracy = 0.833 ± 0.017
Decision_Tree: F1 = 0.822 ± 0.017 | Accuracy = 0.785 ± 0.020
Random_Forest: F1 = 0.856 ± 0.022 | Accuracy = 0.817 ± 0.028
KNN: F1 = 0.821 ± 0.019 | Accuracy = 0.759 ± 0.025


e:\BTL_AI\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
e:\BTL_AI\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
e:\BTL_AI\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
e:\BTL_AI\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=

SVM: F1 = 0.866 ± 0.015 | Accuracy = 0.832 ± 0.021


## Bước 4: Huấn luyện mô hình cuối cùng trên toàn bộ tập Train
Cross-Validation ở Bước 3 chỉ dùng để **đánh giá độ ổn định**. Mô hình dùng để dự đoán thực tế (trong app, trong notebook 04) vẫn cần huấn luyện trên **toàn bộ tập Train** để tận dụng hết dữ liệu có sẵn.

In [4]:
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"Đã huấn luyện xong: {name}")

Đã huấn luyện xong: Dummy_Baseline
Đã huấn luyện xong: Logistic_Regression
Đã huấn luyện xong: Decision_Tree
Đã huấn luyện xong: Random_Forest
Đã huấn luyện xong: KNN
Đã huấn luyện xong: SVM


e:\BTL_AI\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


## Bước 5: Lưu mô hình

In [5]:
import os

os.makedirs("../results/models", exist_ok=True)

for name, model in models.items():
    joblib.dump(model, f"../results/models/{name}.pkl")

print("Đã lưu xong mô hình (bao gồm Dummy Baseline)!")

Đã lưu xong mô hình (bao gồm Dummy Baseline)!
